In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModel,
    BertForTokenClassification,
    BertForQuestionAnswering
)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "dicta-il/dictabert"
MODELS_BASE_DIR = "/content/drive/MyDrive/Workshop/New_Models/"

STAGE1_PATH = os.path.join(MODELS_BASE_DIR, "stage1_dictabert_fix1")
STAGE2_PATH = os.path.join(MODELS_BASE_DIR, "stage2_dictabert")
STAGE3_PATH = os.path.join(MODELS_BASE_DIR, "stage3_dictabert_uncertainty")

STAGE3_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

print("=" * 70)
print(f"LOADING PIPELINE (Device: {device})")
print("=" * 70)

# ============================================================
# 1. LOAD STAGE 1
# ============================================================
print("\n[1/3] Loading Stage 1 Model...")

try:
    tok_s1 = AutoTokenizer.from_pretrained(STAGE1_PATH)
except Exception:
    print("⚠️ Local Stage 1 tokenizer missing/broken. Falling back to dicta-il/dictabert.")
    tok_s1 = AutoTokenizer.from_pretrained(MODEL_NAME)

model_s1 = BertForTokenClassification.from_pretrained(STAGE1_PATH).to(device).eval()
print("✅ Stage 1 fine-tuned model loaded successfully.")


# ============================================================
# 2. LOAD STAGE 2
# ============================================================
print("\n[2/3] Loading Stage 2 Model...")

try:
    tok_s2 = AutoTokenizer.from_pretrained(STAGE2_PATH)
except Exception:
    print("⚠️ Local Stage 2 tokenizer missing/broken. Falling back to dicta-il/dictabert.")
    tok_s2 = AutoTokenizer.from_pretrained(MODEL_NAME)

tok_s2.add_special_tokens({"additional_special_tokens": ["<mv>", "</mv>"]})

model_s2 = BertForQuestionAnswering.from_pretrained(STAGE2_PATH).to(device).eval()
model_s2.resize_token_embeddings(len(tok_s2))
print("✅ Stage 2 fine-tuned model loaded successfully.")


# ============================================================
# 3. LOAD STAGE 3
# ============================================================
print("\n[3/3] Loading Stage 3 Model...")

# Mean Pooling Architecture
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

class DictaBertMultiTaskUncertainty(nn.Module):
    def __init__(self, model_name, num_classes_dict):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)
        self.heads = nn.ModuleDict({
            col: nn.Linear(hidden_size, num_classes_dict[col]) for col in STAGE3_COLS
        })
        self.log_vars = nn.Parameter(torch.zeros(len(STAGE3_COLS)))

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = mean_pooling(outputs, attention_mask)
        pooled_output = self.dropout(pooled_output)
        logits = {col: self.heads[col](pooled_output) for col in STAGE3_COLS}
        return logits

# Load Encoders JSON
encoder_path = os.path.join(STAGE3_PATH, "label_encoders.json")
with open(encoder_path, "r", encoding="utf-8") as f:
    encoders = json.load(f)

decoders = {col: {v: k for k, v in encoders[col].items()} for col in STAGE3_COLS}
num_classes_dict = {col: len(encoders[col]) for col in STAGE3_COLS}

model_s3 = DictaBertMultiTaskUncertainty(MODEL_NAME, num_classes_dict)

s3_weights_path = os.path.join(STAGE3_PATH, "pytorch_model.bin")
if not os.path.exists(s3_weights_path):
    s3_weights_path = os.path.join(STAGE3_PATH, "model.safetensors")

state_dict = torch.load(s3_weights_path, map_location=device) if s3_weights_path.endswith('.bin') else None
if state_dict:
    model_s3.load_state_dict(state_dict, strict=True)

model_s3.to(device).eval()

try:
    tok_s3 = AutoTokenizer.from_pretrained(STAGE3_PATH)
except Exception:
    tok_s3 = AutoTokenizer.from_pretrained(MODEL_NAME)

print("✅ Stage 3 fine-tuned model loaded successfully.")
print("\n🎉 ALL THREE STAGES LOADED AND READY!")

LOADING PIPELINE (Device: cuda)

[1/3] Loading Stage 1 Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Stage 1 fine-tuned model loaded successfully.

[2/3] Loading Stage 2 Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Stage 2 fine-tuned model loaded successfully.

[3/3] Loading Stage 3 Model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Stage 3 fine-tuned model loaded successfully.

🎉 ALL THREE STAGES LOADED AND READY!


In [ ]:
import os
import re
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering
)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "dicta-il/dictabert"

MODELS_BASE_DIR = "/content/drive/MyDrive/Workshop/New_Models/"

STAGE1_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage1_dictabert_fix1/"
STAGE2_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage2_dictabert/"
STAGE3_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty/"

VAL_CSV_PATH = "val_set.csv"

STAGE3_COLS = [
    "dim",
    "tr",
    "fr",
    "sr",
    "fe",
    "refLevel",
    "ss"
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print(f"LOADING CASCADE EVALUATION PIPELINE")
print(f"Device: {device}")
print("=" * 70)


# ------------------------------------------------------------
# Mean pooling
# ------------------------------------------------------------

def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    sum_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sum_embeddings / sum_mask


# ------------------------------------------------------------
# Stage 3 architecture
# ------------------------------------------------------------

class DictaBertMultiTaskUncertainty(torch.nn.Module):

    def __init__(
        self,
        model_name,
        num_classes_dict
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.bert.config.hidden_size

        self.dropout = torch.nn.Dropout(
            self.bert.config.hidden_dropout_prob
        )

        self.heads = torch.nn.ModuleDict({
            col: torch.nn.Linear(
                hidden_size,
                num_classes_dict[col]
            )
            for col in STAGE3_COLS
        })

        self.log_vars = torch.nn.Parameter(
            torch.zeros(len(STAGE3_COLS))
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = mean_pooling(
            outputs,
            attention_mask
        )

        pooled_output = self.dropout(
            pooled_output
        )

        logits = {
            col: self.heads[col](pooled_output)
            for col in STAGE3_COLS
        }

        return logits


# ------------------------------------------------------------
# Recreate Stage 3 model
# ------------------------------------------------------------

model_s3 = DictaBertMultiTaskUncertainty(
    MODEL_NAME,
    num_classes_dict
)


# ------------------------------------------------------------
# Load Stage 3 weights
# ------------------------------------------------------------

s3_weights_path = os.path.join(
    STAGE3_PATH,
    "pytorch_model.bin"
)

if not os.path.exists(s3_weights_path):

    raise FileNotFoundError(
        f"""
Missing Stage 3 weights:

{s3_weights_path}

Make sure the trained Stage 3 model was saved with:
pytorch_model.bin
"""
    )

print(
    f"Loading Stage 3 weights from:\n"
    f"{s3_weights_path}"
)

state_dict = torch.load(
    s3_weights_path,
    map_location=device
)

model_s3.load_state_dict(
    state_dict,
    strict=True
)

model_s3 = model_s3.to(device).eval()

tok_s3 = AutoTokenizer.from_pretrained(
    STAGE3_PATH
)

print("Stage 3 loaded successfully.")

print("\nAll 3 stages successfully loaded.")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_hebrew(text):

    text = str(text)

    # Remove punctuation/symbols.
    text = re.sub(
        r"[^\w\s]",
        "",
        text
    )

    # Normalize whitespace.
    return " ".join(
        text.strip().split()
    )


# ============================================================
# STAGE 1 INFERENCE
# ============================================================

def predict_stage1_mvs(sentence):

    encoding = tok_s1(
        sentence,
        return_tensors="pt",
        max_length=128,
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
    )

    offset_mapping = (
        encoding
        .pop("offset_mapping")[0]
        .cpu()
        .tolist()
    )

    inputs = {
        k: v.to(device)
        for k, v in encoding.items()
    }

    with torch.no_grad():

        outputs = model_s1(
            **inputs
        )

    preds = torch.argmax(
        outputs.logits,
        dim=-1
    )[0].cpu().numpy()

    # --------------------------------------------------------
    # Reconstruct BIO spans
    # --------------------------------------------------------

    spans = []
    curr_span = []

    for idx, (label, (st, end)) in enumerate(
        zip(preds, offset_mapping)
    ):

        # Ignore special tokens / padding.
        if st == 0 and end == 0:
            continue

        if label == 1:
            # B-MU

            if curr_span:
                spans.append(curr_span)

            curr_span = [
                st,
                end
            ]

        elif label == 2:
            # I-MU

            if curr_span:
                curr_span[1] = end

            else:
                curr_span = [
                    st,
                    end
                ]

        else:
            # O

            if curr_span:
                spans.append(curr_span)
                curr_span = []

    if curr_span:
        spans.append(curr_span)

    # --------------------------------------------------------
    # Convert spans into MV objects
    # --------------------------------------------------------

    results = []

    for st, end in spans:

        # Substring from original sentence.
        mv_str = sentence[st:end].strip()

        # Normalized version for evaluation/matching.
        norm_mv = normalize_hebrew(
            mv_str
        )

        if norm_mv:

            results.append({
                "text": mv_str,
                "normalized_text": norm_mv,
                "start_char": st,
                "end_char": end
            })

    return results


# ============================================================
# STAGE 2 INFERENCE
# ============================================================

def predict_stage2_referent(
    sentence,
    mv_info,
    prompt_word
):

    st = mv_info["start_char"]
    end = mv_info["end_char"]

    # Raw substring
    mv_text = mv_info["text"]

    # --------------------------------------------------------
    # Mark the predicted MV inside the original sentence.
    # --------------------------------------------------------

    marked_sentence = (
        f"{sentence[:st]}"
        f"<mv> {mv_text} </mv>"
        f"{sentence[end:]}"
    )

    # Stage 2 training format.
    context = (
        f"גירוי: {prompt_word} | "
        f"משפט: {marked_sentence}"
    )

    question = (
        f"מהו הרפרנט של יחידת המשמעות: "
        f"{mv_text} ?"
    )

    # --------------------------------------------------------
    # Tokenization
    # --------------------------------------------------------

    encoding = tok_s2(
        question,
        context,
        max_length=160,
        padding="max_length",
        truncation="only_second",
        return_offsets_mapping=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in encoding.items()
        if k != "offset_mapping"
    }

    with torch.no_grad():

        outputs = model_s2(
            **inputs
        )

    start_logits = (
        outputs.start_logits[0]
        .cpu()
        .numpy()
    )

    end_logits = (
        outputs.end_logits[0]
        .cpu()
        .numpy()
    )

    token_type_ids = (
        encoding["token_type_ids"][0]
        .cpu()
        .numpy()
    )

    # --------------------------------------------------------
    # Restrict candidate spans to the context.
    # --------------------------------------------------------

    seq_len = len(start_logits)

    context_mask = (
        token_type_ids == 1
    )

    score_matrix = (
        start_logits[:, None]
        +
        end_logits[None, :]
    )

    indices = np.arange(seq_len)

    i_idx, j_idx = np.meshgrid(
        indices,
        indices,
        indexing="ij"
    )

    valid_mask = (
        (i_idx <= j_idx)
        &
        ((j_idx - i_idx + 1) <= 12)
        &
        context_mask[:, None]
        &
        context_mask[None, :]
    )

    score_matrix[
        ~valid_mask
    ] = -1e9

    best_start, best_end = np.unravel_index(
        np.argmax(score_matrix),
        score_matrix.shape
    )

    # --------------------------------------------------------
    # Decode predicted referent
    # --------------------------------------------------------

    input_ids = (
        encoding["input_ids"][0]
        .cpu()
        .tolist()
    )

    pred_tokens = input_ids[
        best_start:best_end + 1
    ]

    raw_ref = tok_s2.decode(
        pred_tokens,
        skip_special_tokens=True
    )

    # Remove MV markers if they appear in decoded text.
    clean_ref = re.sub(
        r"</?mv>",
        "",
        raw_ref
    )

    return normalize_hebrew(
        clean_ref
    )


# ============================================================
# STAGE 3 INFERENCE
# ============================================================

def predict_stage3_codes(
    sentence,
    meaning_val,
    referent
):
    text = (
        f"משפט: {sentence} | "
        f"יחידה: {meaning_val} | "
        f"רפרנט: {referent}"
    )

    encoding = tok_s3(
        text,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    # Filter out token_type_ids or extra keys not explicitly declared in forward()
    inputs = {
        k: v.to(device)
        for k, v in encoding.items()
        if k in ["input_ids", "attention_mask"]
    }

    with torch.no_grad():
        logits = model_s3(**inputs)

    predictions = {}

    for col in STAGE3_COLS:
        pred_idx = torch.argmax(
            logits[col],
            dim=-1
        ).item()

        predictions[col] = (
            decoders[col].get(
                pred_idx,
                "UNKNOWN"
            )
        )

    return predictions


# ============================================================
# EVALUATION RUNNER
# ============================================================

print("\n" + "=" * 70)
print("RUNNING FULL CASCADE EVALUATION")
print("=" * 70)


# ------------------------------------------------------------
# Load validation data
# ------------------------------------------------------------

df_val = pd.read_csv(
    VAL_CSV_PATH
)

grouped = df_val.groupby(
    [
        "Prompt_Word",
        "Full_Raw_Sentence"
    ],
    sort=False
)


# ------------------------------------------------------------
# Metric counters
# ------------------------------------------------------------

# Stage 1
s1_exact_sets = 0
s1_total = 0

# Stage 2
# Conditional on Stage 1 correctly finding the MV.
s2_correct = 0
s2_total = 0

# Stage 3
# Conditional on Stage 1 correctly finding the MV.
s3_correct = {
    col: 0
    for col in STAGE3_COLS
}

s3_total = {
    col: 0
    for col in STAGE3_COLS
}


# ------------------------------------------------------------
# Debug predictions
# ------------------------------------------------------------

all_results = []


for (
    prompt_word,
    sentence
), sentence_df in tqdm(
    grouped,
    desc="Processing Sentences"
):

    sentence = str(
        sentence
    ).strip()

    prompt_word = str(
        prompt_word
    ).strip()


    # ========================================================
    # GOLD MEANING VALUES
    # ========================================================

    gold_mvs = list(
        dict.fromkeys(
            [
                normalize_hebrew(x)
                for x in sentence_df[
                    "MeaningVal"
                ].astype(str)
            ]
        )
    )


    # ========================================================
    # GOLD LOOKUP
    # ========================================================

    gold_lookup = {}

    for _, row in sentence_df.iterrows():

        mv_norm = normalize_hebrew(
            row["MeaningVal"]
        )

        gold_lookup[mv_norm] = {

            "referent": normalize_hebrew(
                row["Referent"]
            ),

            "codes": {
                col: (
                    None
                    if pd.isna(row[col])
                    else str(row[col]).strip()
                )
                for col in STAGE3_COLS
            }
        }


    # ========================================================
    # STAGE 1
    # ========================================================

    predicted_mv_infos = (
        predict_stage1_mvs(
            sentence
        )
    )


    # --------------------------------------------------------
    # Deduplicate predicted MVs.
    # --------------------------------------------------------

    seen = set()

    unique_mv_infos = []

    for item in predicted_mv_infos:

        norm_mv = item[
            "normalized_text"
        ]

        if norm_mv not in seen:

            seen.add(norm_mv)

            unique_mv_infos.append(
                item
            )


    # --------------------------------------------------------
    # Stage 1 prediction for evaluation
    # --------------------------------------------------------

    pred_mv_texts = [
        item["normalized_text"]
        for item in unique_mv_infos
    ]


    # Exact set match.
    if set(pred_mv_texts) == set(gold_mvs):

        s1_exact_sets += 1

    s1_total += 1


    # ========================================================
    # STAGE 2 + STAGE 3
    # ========================================================

    for mv_info in unique_mv_infos:

        # ----------------------------------------------------
        # Stage 2
        # ----------------------------------------------------

        pred_ref = predict_stage2_referent(
            sentence,
            mv_info,
            prompt_word
        )


        # Normalized MV for matching/evaluation.
        pred_mv_norm = (
            mv_info["normalized_text"]
        )


        # ----------------------------------------------------
        # Find corresponding gold MV.
        # ----------------------------------------------------

        gold_match = gold_lookup.get(
            pred_mv_norm
        )


        # ====================================================
        # STAGE 2 EVALUATION
        # ====================================================

        if gold_match is not None:

            if (
                pred_ref
                ==
                gold_match["referent"]
            ):
                s2_correct += 1

            s2_total += 1


        # ====================================================
        # STAGE 3
        # ====================================================

        pred_codes = predict_stage3_codes(
            sentence,
            pred_mv_norm,
            pred_ref
        )


        # ====================================================
        # STAGE 3 EVALUATION
        # ====================================================

        if gold_match is not None:

            for col in STAGE3_COLS:

                gold_val = (
                    gold_match["codes"][col]
                )

                if gold_val is not None:

                    s3_total[col] += 1

                    if (
                        pred_codes[col]
                        ==
                        gold_val
                    ):
                        s3_correct[col] += 1


        # ====================================================
        # SAVE DEBUG INFORMATION
        # ====================================================

        all_results.append({

            "sentence": sentence,

            # Raw Stage 1 substring
            "predicted_mv_raw": mv_info[
                "text"
            ],

            # Normalized Stage 1 substring
            "predicted_mv_norm": pred_mv_norm,

            # Stage 2 prediction
            "predicted_referent": pred_ref,

            # Stage 3 predictions
            **{
                f"pred_{col}": pred_codes[col]
                for col in STAGE3_COLS
            }
        })


# ============================================================
# RESULTS SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL CASCADE METRICS")
print("=" * 70)


# ============================================================
# STAGE 1 RESULTS
# ============================================================

print("\nStage 1 - Meaning Value Segmentation:")

print(
    f"  Exact Set Match Rate : "
    f"{100 * s1_exact_sets / max(s1_total, 1):.2f}% "
    f"({s1_exact_sets}/{s1_total})"
)


# ============================================================
# STAGE 2 RESULTS
# ============================================================

print("\nStage 2 - Referent Extraction:")

print(
    "  Exact Match Accuracy : "
    f"{100 * s2_correct / max(s2_total, 1):.2f}% "
    f"({s2_correct}/{s2_total})"
)

print(
    "  NOTE: This is conditional on Stage 1 "
    "correctly predicting the Meaning Value."
)


# ============================================================
# STAGE 3 RESULTS
# ============================================================

print(
    "\nStage 3 - Psychological Attribute "
    "Multi-Task Classification:"
)

for col in STAGE3_COLS:

    tot = s3_total[col]

    acc = (
        100 * s3_correct[col] / tot
        if tot > 0
        else 0.0
    )

    print(
        f"  {col:10s} Accuracy : "
        f"{acc:.2f}% "
        f"({s3_correct[col]}/{tot})"
    )

print(
    "\n  NOTE: Stage 3 metrics are conditional on "
    "Stage 1 correctly predicting the Meaning Value."
)

print(
    "  Stage 3 receives the referent predicted by Stage 2."
)


# ============================================================
# SAVE DEBUG OUTPUT
# ============================================================

output_path = "cascade_predictions_debug.csv"

pd.DataFrame(
    all_results
).to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nSaved detailed cascade outputs to: "
    f"{output_path}"
)

print("\n" + "=" * 70)
print("CASCADE EVALUATION COMPLETE")
print("=" * 70)

LOADING CASCADE EVALUATION PIPELINE
Device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading Stage 3 weights from:
/content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty/pytorch_model.bin
Stage 3 loaded successfully.

All 3 stages successfully loaded.

RUNNING FULL CASCADE EVALUATION


Processing Sentences: 100%|██████████| 808/808 [02:05<00:00,  6.45it/s]



FINAL CASCADE METRICS

Stage 1 - Meaning Value Segmentation:
  Exact Set Match Rate : 39.36% (318/808)

Stage 2 - Referent Extraction:
  Exact Match Accuracy : 73.28% (2284/3117)
  NOTE: This is conditional on Stage 1 correctly predicting the Meaning Value.

Stage 3 - Psychological Attribute Multi-Task Classification:
  dim        Accuracy : 79.65% (2477/3110)
  tr         Accuracy : 85.24% (2651/3110)
  fr         Accuracy : 96.14% (2990/3110)
  sr         Accuracy : 87.41% (2721/3113)
  fe         Accuracy : 99.94% (3107/3109)
  refLevel   Accuracy : 69.44% (2161/3112)
  ss         Accuracy : 87.64% (78/89)

  NOTE: Stage 3 metrics are conditional on Stage 1 correctly predicting the Meaning Value.
  Stage 3 receives the referent predicted by Stage 2.

Saved detailed cascade outputs to: cascade_predictions_debug.csv

CASCADE EVALUATION COMPLETE


Evaluation

In [ ]:
# ============================================================
# CASCADE METRICS & FUNNEL ANALYSIS
# ============================================================

df_val = pd.read_csv(VAL_CSV_PATH)
grouped = df_val.groupby(["Prompt_Word", "Full_Raw_Sentence"], sort=False)

# Tracking Counts
total_val_sentences = len(grouped)
total_gold_mvs = len(df_val)

s1_exact_sentence_matches = 0
s1_extracted_mvs_total = 0
s1_correct_mvs_matched = 0

s2_evaluated_mvs = 0
s2_exact_referent_matches = 0

s3_evaluated_head_counts = {col: 0 for col in STAGE3_COLS}
s3_correct_head_counts = {col: 0 for col in STAGE3_COLS}

for (prompt_word, sentence), sentence_df in tqdm(grouped, desc="Evaluating Pipeline Funnel"):
    sentence, prompt_word = str(sentence).strip(), str(prompt_word).strip()

    # Ground Truth MVs
    gold_mvs = list(dict.fromkeys([normalize_hebrew(x) for x in sentence_df["MeaningVal"].astype(str)]))
    gold_lookup = {
        normalize_hebrew(row["MeaningVal"]): {
            "referent": normalize_hebrew(row["Referent"]),
            "codes": {col: (None if pd.isna(row[col]) else str(row[col]).strip()) for col in STAGE3_COLS}
        }
        for _, row in sentence_df.iterrows()
    }

    # --- STAGE 1 ---
    predicted_mv_infos = predict_stage1_mvs(sentence)

    seen = set()
    unique_mv_infos = [x for x in predicted_mv_infos if not (x["normalized_text"] in seen or seen.add(x["normalized_text"]))]
    pred_mv_norms = [x["normalized_text"] for x in unique_mv_infos]

    s1_extracted_mvs_total += len(unique_mv_infos)

    if set(pred_mv_norms) == set(gold_mvs):
        s1_exact_sentence_matches += 1

    for norm_mv in pred_mv_norms:
        if norm_mv in gold_lookup:
            s1_correct_mvs_matched += 1

    # --- STAGES 2 & 3 ---
    for mv_info in unique_mv_infos:
        pred_ref = predict_stage2_referent(sentence, mv_info, prompt_word)
        norm_mv = mv_info["normalized_text"]

        gold_match = gold_lookup.get(norm_mv, None)

        if gold_match is not None:
            s2_evaluated_mvs += 1
            if pred_ref == gold_match["referent"]:
                s2_exact_referent_matches += 1

        pred_codes = predict_stage3_codes(sentence, norm_mv, pred_ref)

        if gold_match is not None:
            for col in STAGE3_COLS:
                gold_val = gold_match["codes"][col]
                if gold_val is not None:
                    s3_evaluated_head_counts[col] += 1
                    if pred_codes[col] == gold_val:
                        s3_correct_head_counts[col] += 1

# ============================================================
# PIPELINE FUNNEL & METRICS REPORT
# ============================================================
print("\n" + "=" * 75)
print("                    PIPELINE FUNNEL & ACCURACY METRICS")
print("=" * 75)

print("\n🔻 [FUNNEL TRACKING]")
print(f"  • Total Sentences Input             : {total_val_sentences}")
print(f"  • Sentences with 100% Correct MVs  : {s1_exact_sentence_matches} ({100*s1_exact_sentence_matches/total_val_sentences:.1f}%)")
print(f"  • Total Ground-Truth MVs            : {total_gold_mvs}")
print(f"  • Total MVs Extracted by Stage 1    : {s1_extracted_mvs_total}")
print(f"  • Extracted MVs Matching Ground Truth: {s1_correct_mvs_matched} ({100*s1_correct_mvs_matched/max(s1_extracted_mvs_total,1):.1f}%)")
print(f"  • MVs Advanced to Stage 2 Evaluation: {s2_evaluated_mvs}")

print("\n🎯 [MODEL METRICS]")
print("STAGE 1 (Segmentation):")
print(f"  • Sentence Exact-Set Match Accuracy : {100 * s1_exact_sentence_matches / total_val_sentences:.2f}%")
print(f"  • Individual MV Precision           : {100 * s1_correct_mvs_matched / max(s1_extracted_mvs_total, 1):.2f}%")
print(f"  • Individual MV Recall              : {100 * s1_correct_mvs_matched / total_gold_mvs:.2f}%")

print("\nSTAGE 2 (Referent Extraction):")
print(f"  • Exact Match Referent Accuracy     : {100 * s2_exact_referent_matches / max(s2_evaluated_mvs, 1):.2f}% ({s2_exact_referent_matches}/{s2_evaluated_mvs})")

print("\nSTAGE 3 (Kreitler Psychological Dimensions):")
for col in STAGE3_COLS:
    tot = s3_evaluated_head_counts[col]
    acc = (100 * s3_correct_head_counts[col] / tot) if tot > 0 else 0.0
    print(f"  • Head [{col:8s}] Accuracy          : {acc:.2f}% ({s3_correct_head_counts[col]}/{tot})")
print("=" * 75)

Evaluating Pipeline Funnel: 100%|██████████| 808/808 [02:06<00:00,  6.39it/s]


                    PIPELINE FUNNEL & ACCURACY METRICS

🔻 [FUNNEL TRACKING]
  • Total Sentences Input             : 808
  • Sentences with 100% Correct MVs  : 318 (39.4%)
  • Total Ground-Truth MVs            : 4569
  • Total MVs Extracted by Stage 1    : 4322
  • Extracted MVs Matching Ground Truth: 3117 (72.1%)
  • MVs Advanced to Stage 2 Evaluation: 3117

🎯 [MODEL METRICS]
STAGE 1 (Segmentation):
  • Sentence Exact-Set Match Accuracy : 39.36%
  • Individual MV Precision           : 72.12%
  • Individual MV Recall              : 68.22%

STAGE 2 (Referent Extraction):
  • Exact Match Referent Accuracy     : 73.28% (2284/3117)

STAGE 3 (Kreitler Psychological Dimensions):
  • Head [dim     ] Accuracy          : 79.65% (2477/3110)
  • Head [tr      ] Accuracy          : 85.24% (2651/3110)
  • Head [fr      ] Accuracy          : 96.14% (2990/3110)
  • Head [sr      ] Accuracy          : 87.41% (2721/3113)
  • Head [fe      ] Accuracy          : 99.94% (3107/3109)
  • Head [refLevel] Acc

One Sentence Analysis

In [ ]:
# ============================================================
# SINGLE-SENTENCE PIPELINE FUNCTION
# ============================================================

def analyze_sentence(prompt_word: str, sentence: str):
    """
    Passes a raw Hebrew sentence through Stage 1 -> Stage 2 -> Stage 3.
    """
    sentence = sentence.strip()
    prompt_word = prompt_word.strip()

    # --- STAGE 1: Extract Meaning Values ---
    mv_infos = predict_stage1_mvs(sentence)

    # Deduplicate MVs while preserving order
    seen = set()
    unique_mvs = [
        item for item in mv_infos
        if not (item["normalized_text"] in seen or seen.add(item["normalized_text"]))
    ]

    if not unique_mvs:
        print("⚠️ Stage 1 found no Meaning Values in this sentence.")
        return pd.DataFrame()

    results = []

    # --- STAGES 2 & 3: Extract Referents and Predict Codes ---
    for mv_info in unique_mvs:
        # Stage 2: Extract Referent using raw character positions
        pred_ref = predict_stage2_referent(sentence, mv_info, prompt_word)

        # Stage 3: Classify Psychological Codes
        pred_codes = predict_stage3_codes(sentence, mv_info["normalized_text"], pred_ref)

        # Build clean output row
        row = {
            "Prompt (גירוי)": prompt_word,
            "Full Sentence (משפט)": sentence,
            "Meaning Value (יחידת משמעות)": mv_info["text"],
            "Referent (רפרנט)": pred_ref,
            **pred_codes
        }
        results.append(row)

    return pd.DataFrame(results)

In [ ]:
# Test Example 1
df_res = analyze_sentence("רחוב", "הכתובת שאני שמה בוייז כדי להגיע לבית")
df_res

,Prompt (גירוי),Full Sentence (משפט),Meaning Value (יחידת משמעות),Referent (רפרנט),dim,tr,fr,sr,fe,refLevel,ss
0,רחוב,הכתובת שאני שמה בוייז כדי להגיע לבית,הכתובת,רחוב,1,1a,1a,1,1a,1.0,3.0
1,רחוב,הכתובת שאני שמה בוייז כדי להגיע לבית,שאני,הכתובת,8a,1a,1a,5,1a,2.0,3.0
2,רחוב,הכתובת שאני שמה בוייז כדי להגיע לבית,שמה,שאני,4a,1b,1a,5,1a,3.0,1.0
3,רחוב,הכתובת שאני שמה בוייז כדי להגיע לבית,בוייז,שמה,8b,1a,1a,5,1a,4.0,11.0
4,רחוב,הכתובת שאני שמה בוייז כדי להגיע לבית,כדי להגיע,שמה,3,1a,1a,5,1a,4.0,11.0
5,רחוב,הכתובת שאני שמה בוייז כדי להגיע לבית,לבית,להגיע,15,1a,1a,5,1a,5.0,11.0


same thing but on test

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import json
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModel,
    BertForTokenClassification,
    BertForQuestionAnswering
)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "dicta-il/dictabert"
MODELS_BASE_DIR = "/content/drive/MyDrive/Workshop/New_Models/"

STAGE1_PATH = os.path.join(MODELS_BASE_DIR, "stage1_dictabert_fix1")
STAGE2_PATH = os.path.join(MODELS_BASE_DIR, "stage2_dictabert")
STAGE3_PATH = os.path.join(MODELS_BASE_DIR, "stage3_dictabert_uncertainty")

STAGE3_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

print("=" * 70)
print(f"LOADING PIPELINE (Device: {device})")
print("=" * 70)

# ============================================================
# 1. LOAD STAGE 1
# ============================================================
print("\n[1/3] Loading Stage 1 Model...")

try:
    tok_s1 = AutoTokenizer.from_pretrained(STAGE1_PATH)
except Exception:
    print("⚠️ Local Stage 1 tokenizer missing/broken. Falling back to dicta-il/dictabert.")
    tok_s1 = AutoTokenizer.from_pretrained(MODEL_NAME)

model_s1 = BertForTokenClassification.from_pretrained(STAGE1_PATH).to(device).eval()
print("✅ Stage 1 fine-tuned model loaded successfully.")


# ============================================================
# 2. LOAD STAGE 2
# ============================================================
print("\n[2/3] Loading Stage 2 Model...")

try:
    tok_s2 = AutoTokenizer.from_pretrained(STAGE2_PATH)
except Exception:
    print("⚠️ Local Stage 2 tokenizer missing/broken. Falling back to dicta-il/dictabert.")
    tok_s2 = AutoTokenizer.from_pretrained(MODEL_NAME)

tok_s2.add_special_tokens({"additional_special_tokens": ["<mv>", "</mv>"]})

model_s2 = BertForQuestionAnswering.from_pretrained(STAGE2_PATH).to(device).eval()
model_s2.resize_token_embeddings(len(tok_s2))
print("✅ Stage 2 fine-tuned model loaded successfully.")


# ============================================================
# 3. LOAD STAGE 3
# ============================================================
print("\n[3/3] Loading Stage 3 Model...")

# Mean Pooling Architecture
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

class DictaBertMultiTaskUncertainty(nn.Module):
    def __init__(self, model_name, num_classes_dict):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)
        self.heads = nn.ModuleDict({
            col: nn.Linear(hidden_size, num_classes_dict[col]) for col in STAGE3_COLS
        })
        self.log_vars = nn.Parameter(torch.zeros(len(STAGE3_COLS)))

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = mean_pooling(outputs, attention_mask)
        pooled_output = self.dropout(pooled_output)
        logits = {col: self.heads[col](pooled_output) for col in STAGE3_COLS}
        return logits

# Load Encoders JSON
encoder_path = os.path.join(STAGE3_PATH, "label_encoders.json")
with open(encoder_path, "r", encoding="utf-8") as f:
    encoders = json.load(f)

decoders = {col: {v: k for k, v in encoders[col].items()} for col in STAGE3_COLS}
num_classes_dict = {col: len(encoders[col]) for col in STAGE3_COLS}

model_s3 = DictaBertMultiTaskUncertainty(MODEL_NAME, num_classes_dict)

s3_weights_path = os.path.join(STAGE3_PATH, "pytorch_model.bin")
if not os.path.exists(s3_weights_path):
    s3_weights_path = os.path.join(STAGE3_PATH, "model.safetensors")

state_dict = torch.load(s3_weights_path, map_location=device) if s3_weights_path.endswith('.bin') else None
if state_dict:
    model_s3.load_state_dict(state_dict, strict=True)

model_s3.to(device).eval()

try:
    tok_s3 = AutoTokenizer.from_pretrained(STAGE3_PATH)
except Exception:
    tok_s3 = AutoTokenizer.from_pretrained(MODEL_NAME)

print("✅ Stage 3 fine-tuned model loaded successfully.")
print("\n🎉 ALL THREE STAGES LOADED AND READY!")

LOADING PIPELINE (Device: cuda)

[1/3] Loading Stage 1 Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Stage 1 fine-tuned model loaded successfully.

[2/3] Loading Stage 2 Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Stage 2 fine-tuned model loaded successfully.

[3/3] Loading Stage 3 Model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Stage 3 fine-tuned model loaded successfully.

🎉 ALL THREE STAGES LOADED AND READY!


On test

In [ ]:
import os
import re
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering
)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "dicta-il/dictabert"

MODELS_BASE_DIR = "/content/drive/MyDrive/Workshop/New_Models/"

STAGE1_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage1_dictabert_fix1/"
STAGE2_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage2_dictabert/"
STAGE3_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty/"

VAL_CSV_PATH = "test_set.csv"

STAGE3_COLS = [
    "dim",
    "tr",
    "fr",
    "sr",
    "fe",
    "refLevel",
    "ss"
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print(f"LOADING CASCADE EVALUATION PIPELINE")
print(f"Device: {device}")
print("=" * 70)


# ------------------------------------------------------------
# Mean pooling
# ------------------------------------------------------------

def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    sum_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sum_embeddings / sum_mask


# ------------------------------------------------------------
# Stage 3 architecture
# ------------------------------------------------------------

class DictaBertMultiTaskUncertainty(torch.nn.Module):

    def __init__(
        self,
        model_name,
        num_classes_dict
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.bert.config.hidden_size

        self.dropout = torch.nn.Dropout(
            self.bert.config.hidden_dropout_prob
        )

        self.heads = torch.nn.ModuleDict({
            col: torch.nn.Linear(
                hidden_size,
                num_classes_dict[col]
            )
            for col in STAGE3_COLS
        })

        self.log_vars = torch.nn.Parameter(
            torch.zeros(len(STAGE3_COLS))
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = mean_pooling(
            outputs,
            attention_mask
        )

        pooled_output = self.dropout(
            pooled_output
        )

        logits = {
            col: self.heads[col](pooled_output)
            for col in STAGE3_COLS
        }

        return logits


# ------------------------------------------------------------
# Stage 3 model
# ------------------------------------------------------------

model_s3 = DictaBertMultiTaskUncertainty(
    MODEL_NAME,
    num_classes_dict
)


# ------------------------------------------------------------
# Load Stage 3 weights
# ------------------------------------------------------------

s3_weights_path = os.path.join(
    STAGE3_PATH,
    "pytorch_model.bin"
)

if not os.path.exists(s3_weights_path):

    raise FileNotFoundError(
        f"""
Missing Stage 3 weights:

{s3_weights_path}

Make sure the trained Stage 3 model was saved with:
pytorch_model.bin
"""
    )

print(
    f"Loading Stage 3 weights from:\n"
    f"{s3_weights_path}"
)

state_dict = torch.load(
    s3_weights_path,
    map_location=device
)

model_s3.load_state_dict(
    state_dict,
    strict=True
)

model_s3 = model_s3.to(device).eval()

tok_s3 = AutoTokenizer.from_pretrained(
    STAGE3_PATH
)

print("Stage 3 loaded successfully.")

print("\nAll 3 stages successfully loaded.")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_hebrew(text):

    text = str(text)

    # Remove punctuation/symbols.
    text = re.sub(
        r"[^\w\s]",
        "",
        text
    )

    # Normalize whitespace.
    return " ".join(
        text.strip().split()
    )


# ============================================================
# STAGE 1 INFERENCE
# ============================================================

def predict_stage1_mvs(sentence):

    encoding = tok_s1(
        sentence,
        return_tensors="pt",
        max_length=128,
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
    )

    offset_mapping = (
        encoding
        .pop("offset_mapping")[0]
        .cpu()
        .tolist()
    )

    inputs = {
        k: v.to(device)
        for k, v in encoding.items()
    }

    with torch.no_grad():

        outputs = model_s1(
            **inputs
        )

    preds = torch.argmax(
        outputs.logits,
        dim=-1
    )[0].cpu().numpy()

    # --------------------------------------------------------
    # Reconstruct BIO spans
    # --------------------------------------------------------

    spans = []
    curr_span = []

    for idx, (label, (st, end)) in enumerate(
        zip(preds, offset_mapping)
    ):

        # Ignore special tokens / padding.
        if st == 0 and end == 0:
            continue

        if label == 1:
            # B-MU

            if curr_span:
                spans.append(curr_span)

            curr_span = [
                st,
                end
            ]

        elif label == 2:
            # I-MU

            if curr_span:
                curr_span[1] = end

            else:.
                curr_span = [
                    st,
                    end
                ]

        else:
            # O

            if curr_span:
                spans.append(curr_span)
                curr_span = []

    if curr_span:
        spans.append(curr_span)

    # --------------------------------------------------------
    # Convert spans into MV objects
    # --------------------------------------------------------

    results = []

    for st, end in spans:

        # Substring from original sentence.
        mv_str = sentence[st:end].strip()

        # Normalized version for evaluation/matching.
        norm_mv = normalize_hebrew(
            mv_str
        )

        if norm_mv:

            results.append({
                "text": mv_str,
                "normalized_text": norm_mv,
                "start_char": st,
                "end_char": end
            })

    return results


# ============================================================
# STAGE 2 INFERENCE
# ============================================================

def predict_stage2_referent(
    sentence,
    mv_info,
    prompt_word
):

    st = mv_info["start_char"]
    end = mv_info["end_char"]

    # Raw substring
    mv_text = mv_info["text"]

    # --------------------------------------------------------
    # Mark the predicted MV inside the original sentence.
    # --------------------------------------------------------

    marked_sentence = (
        f"{sentence[:st]}"
        f"<mv> {mv_text} </mv>"
        f"{sentence[end:]}"
    )

    # Stage 2 training format.
    context = (
        f"גירוי: {prompt_word} | "
        f"משפט: {marked_sentence}"
    )

    question = (
        f"מהו הרפרנט של יחידת המשמעות: "
        f"{mv_text} ?"
    )

    # --------------------------------------------------------
    # Tokenization
    # --------------------------------------------------------

    encoding = tok_s2(
        question,
        context,
        max_length=160,
        padding="max_length",
        truncation="only_second",
        return_offsets_mapping=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in encoding.items()
        if k != "offset_mapping"
    }

    with torch.no_grad():

        outputs = model_s2(
            **inputs
        )

    start_logits = (
        outputs.start_logits[0]
        .cpu()
        .numpy()
    )

    end_logits = (
        outputs.end_logits[0]
        .cpu()
        .numpy()
    )

    token_type_ids = (
        encoding["token_type_ids"][0]
        .cpu()
        .numpy()
    )

    # --------------------------------------------------------
    # Restrict candidate spans to the context.
    # --------------------------------------------------------

    seq_len = len(start_logits)

    context_mask = (
        token_type_ids == 1
    )

    score_matrix = (
        start_logits[:, None]
        +
        end_logits[None, :]
    )

    indices = np.arange(seq_len)

    i_idx, j_idx = np.meshgrid(
        indices,
        indices,
        indexing="ij"
    )

    valid_mask = (
        (i_idx <= j_idx)
        &
        ((j_idx - i_idx + 1) <= 12)
        &
        context_mask[:, None]
        &
        context_mask[None, :]
    )

    score_matrix[
        ~valid_mask
    ] = -1e9

    best_start, best_end = np.unravel_index(
        np.argmax(score_matrix),
        score_matrix.shape
    )

    # --------------------------------------------------------
    # Decode predicted referent
    # --------------------------------------------------------

    input_ids = (
        encoding["input_ids"][0]
        .cpu()
        .tolist()
    )

    pred_tokens = input_ids[
        best_start:best_end + 1
    ]

    raw_ref = tok_s2.decode(
        pred_tokens,
        skip_special_tokens=True
    )

    # Remove MV markers if they appear in decoded text.
    clean_ref = re.sub(
        r"</?mv>",
        "",
        raw_ref
    )

    return normalize_hebrew(
        clean_ref
    )


# ============================================================
# STAGE 3 INFERENCE
# ============================================================

def predict_stage3_codes(
    sentence,
    meaning_val,
    referent
):
    text = (
        f"משפט: {sentence} | "
        f"יחידה: {meaning_val} | "
        f"רפרנט: {referent}"
    )

    encoding = tok_s3(
        text,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    # Filter out token_type_ids or extra keys not explicitly declared in forward()
    inputs = {
        k: v.to(device)
        for k, v in encoding.items()
        if k in ["input_ids", "attention_mask"]
    }

    with torch.no_grad():
        logits = model_s3(**inputs)

    predictions = {}

    for col in STAGE3_COLS:
        pred_idx = torch.argmax(
            logits[col],
            dim=-1
        ).item()

        predictions[col] = (
            decoders[col].get(
                pred_idx,
                "UNKNOWN"
            )
        )

    return predictions


# ============================================================
# EVALUATION RUNNER
# ============================================================

print("\n" + "=" * 70)
print("RUNNING FULL CASCADE EVALUATION")
print("=" * 70)


# ------------------------------------------------------------
# Load validation data
# ------------------------------------------------------------

df_val = pd.read_csv(
    VAL_CSV_PATH
)

grouped = df_val.groupby(
    [
        "Prompt_Word",
        "Full_Raw_Sentence"
    ],
    sort=False
)


# ------------------------------------------------------------
# Metric counters
# ------------------------------------------------------------

# Stage 1
s1_exact_sets = 0
s1_total = 0

# Stage 2
s2_correct = 0
s2_total = 0

# Stage 3
s3_correct = {
    col: 0
    for col in STAGE3_COLS
}

s3_total = {
    col: 0
    for col in STAGE3_COLS
}


# ------------------------------------------------------------
# Debug predictions
# ------------------------------------------------------------

all_results = []


for (
    prompt_word,
    sentence
), sentence_df in tqdm(
    grouped,
    desc="Processing Sentences"
):

    sentence = str(
        sentence
    ).strip()

    prompt_word = str(
        prompt_word
    ).strip()


    # ========================================================
    # GOLD MEANING VALUES
    # ========================================================

    gold_mvs = list(
        dict.fromkeys(
            [
                normalize_hebrew(x)
                for x in sentence_df[
                    "MeaningVal"
                ].astype(str)
            ]
        )
    )


    # ========================================================
    # GOLD LOOKUP
    # ========================================================

    gold_lookup = {}

    for _, row in sentence_df.iterrows():

        mv_norm = normalize_hebrew(
            row["MeaningVal"]
        )

        gold_lookup[mv_norm] = {

            "referent": normalize_hebrew(
                row["Referent"]
            ),

            "codes": {
                col: (
                    None
                    if pd.isna(row[col])
                    else str(row[col]).strip()
                )
                for col in STAGE3_COLS
            }
        }


    # ========================================================
    # STAGE 1
    # ========================================================

    predicted_mv_infos = (
        predict_stage1_mvs(
            sentence
        )
    )


    # --------------------------------------------------------
    # Deduplicate predicted MVs.
    # --------------------------------------------------------

    seen = set()

    unique_mv_infos = []

    for item in predicted_mv_infos:

        norm_mv = item[
            "normalized_text"
        ]

        if norm_mv not in seen:

            seen.add(norm_mv)

            unique_mv_infos.append(
                item
            )


    # --------------------------------------------------------
    # Stage 1 prediction for evaluation
    # --------------------------------------------------------

    pred_mv_texts = [
        item["normalized_text"]
        for item in unique_mv_infos
    ]


    # Exact set match.
    if set(pred_mv_texts) == set(gold_mvs):

        s1_exact_sets += 1

    s1_total += 1


    # ========================================================
    # STAGE 2 + STAGE 3
    # ========================================================

    for mv_info in unique_mv_infos:

        # ----------------------------------------------------
        # Stage 2
        # ----------------------------------------------------

        pred_ref = predict_stage2_referent(
            sentence,
            mv_info,
            prompt_word
        )


        # Normalized MV for matching/evaluation.
        pred_mv_norm = (
            mv_info["normalized_text"]
        )


        # ----------------------------------------------------
        # Find corresponding gold MV.
        # ----------------------------------------------------

        gold_match = gold_lookup.get(
            pred_mv_norm
        )


        # ====================================================
        # STAGE 2 EVALUATION
        # ====================================================

        if gold_match is not None:

            if (
                pred_ref
                ==
                gold_match["referent"]
            ):
                s2_correct += 1

            s2_total += 1


        # ====================================================
        # STAGE 3
        # ====================================================

        pred_codes = predict_stage3_codes(
            sentence,
            pred_mv_norm,
            pred_ref
        )


        # ====================================================
        # STAGE 3 EVALUATION
        # ====================================================

        if gold_match is not None:

            for col in STAGE3_COLS:

                gold_val = (
                    gold_match["codes"][col]
                )

                if gold_val is not None:

                    s3_total[col] += 1

                    if (
                        pred_codes[col]
                        ==
                        gold_val
                    ):
                        s3_correct[col] += 1


        # ====================================================
        # SAVE DEBUG INFORMATION
        # ====================================================

        all_results.append({

            "sentence": sentence,

            # Raw Stage 1 substring
            "predicted_mv_raw": mv_info[
                "text"
            ],

            # Normalized Stage 1 substring
            "predicted_mv_norm": pred_mv_norm,

            # Stage 2 prediction
            "predicted_referent": pred_ref,

            # Stage 3 predictions
            **{
                f"pred_{col}": pred_codes[col]
                for col in STAGE3_COLS
            }
        })


# ============================================================
# RESULTS SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL CASCADE METRICS")
print("=" * 70)


# ============================================================
# STAGE 1 RESULTS
# ============================================================

print("\nStage 1 - Meaning Value Segmentation:")

print(
    f"  Exact Set Match Rate : "
    f"{100 * s1_exact_sets / max(s1_total, 1):.2f}% "
    f"({s1_exact_sets}/{s1_total})"
)


# ============================================================
# STAGE 2 RESULTS
# ============================================================

print("\nStage 2 - Referent Extraction:")

print(
    "  Exact Match Accuracy : "
    f"{100 * s2_correct / max(s2_total, 1):.2f}% "
    f"({s2_correct}/{s2_total})"
)

print(
    "  NOTE: This is conditional on Stage 1 "
    "correctly predicting the Meaning Value."
)


# ============================================================
# STAGE 3 RESULTS
# ============================================================

print(
    "\nStage 3 - Psychological Attribute "
    "Multi-Task Classification:"
)

for col in STAGE3_COLS:

    tot = s3_total[col]

    acc = (
        100 * s3_correct[col] / tot
        if tot > 0
        else 0.0
    )

    print(
        f"  {col:10s} Accuracy : "
        f"{acc:.2f}% "
        f"({s3_correct[col]}/{tot})"
    )

print(
    "\n  NOTE: Stage 3 metrics are conditional on "
    "Stage 1 correctly predicting the Meaning Value."
)

print(
    "  Stage 3 receives the referent predicted by Stage 2."
)


# ============================================================
# SAVE DEBUG OUTPUT
# ============================================================

output_path = "cascade_predictions_debug.csv"

pd.DataFrame(
    all_results
).to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nSaved detailed cascade outputs to: "
    f"{output_path}"
)

print("\n" + "=" * 70)
print("CASCADE EVALUATION COMPLETE")
print("=" * 70)

LOADING CASCADE EVALUATION PIPELINE
Device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading Stage 3 weights from:
/content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty/pytorch_model.bin
Stage 3 loaded successfully.

All 3 stages successfully loaded.

RUNNING FULL CASCADE EVALUATION


Processing Sentences: 100%|██████████| 800/800 [01:56<00:00,  6.86it/s]



FINAL CASCADE METRICS

Stage 1 - Meaning Value Segmentation:
  Exact Set Match Rate : 41.88% (335/800)

Stage 2 - Referent Extraction:
  Exact Match Accuracy : 73.70% (2105/2856)
  NOTE: This is conditional on Stage 1 correctly predicting the Meaning Value.

Stage 3 - Psychological Attribute Multi-Task Classification:
  dim        Accuracy : 77.63% (2214/2852)
  tr         Accuracy : 86.43% (2465/2852)
  fr         Accuracy : 96.74% (2759/2852)
  sr         Accuracy : 88.06% (2515/2856)
  fe         Accuracy : 99.93% (2851/2853)
  refLevel   Accuracy : 71.90% (2052/2854)
  ss         Accuracy : 85.25% (52/61)

  NOTE: Stage 3 metrics are conditional on Stage 1 correctly predicting the Meaning Value.
  Stage 3 receives the referent predicted by Stage 2.

Saved detailed cascade outputs to: cascade_predictions_debug.csv

CASCADE EVALUATION COMPLETE


In [ ]:
# ============================================================
# CASCADE METRICS & FUNNEL ANALYSIS
# ============================================================

df_val = pd.read_csv(VAL_CSV_PATH)
grouped = df_val.groupby(["Prompt_Word", "Full_Raw_Sentence"], sort=False)

# Tracking Counts
total_val_sentences = len(grouped)
total_gold_mvs = len(df_val)

s1_exact_sentence_matches = 0
s1_extracted_mvs_total = 0
s1_correct_mvs_matched = 0

s2_evaluated_mvs = 0
s2_exact_referent_matches = 0

s3_evaluated_head_counts = {col: 0 for col in STAGE3_COLS}
s3_correct_head_counts = {col: 0 for col in STAGE3_COLS}

for (prompt_word, sentence), sentence_df in tqdm(grouped, desc="Evaluating Pipeline Funnel"):
    sentence, prompt_word = str(sentence).strip(), str(prompt_word).strip()

    # Ground Truth MVs
    gold_mvs = list(dict.fromkeys([normalize_hebrew(x) for x in sentence_df["MeaningVal"].astype(str)]))
    gold_lookup = {
        normalize_hebrew(row["MeaningVal"]): {
            "referent": normalize_hebrew(row["Referent"]),
            "codes": {col: (None if pd.isna(row[col]) else str(row[col]).strip()) for col in STAGE3_COLS}
        }
        for _, row in sentence_df.iterrows()
    }

    # --- STAGE 1 ---
    predicted_mv_infos = predict_stage1_mvs(sentence)

    seen = set()
    unique_mv_infos = [x for x in predicted_mv_infos if not (x["normalized_text"] in seen or seen.add(x["normalized_text"]))]
    pred_mv_norms = [x["normalized_text"] for x in unique_mv_infos]

    s1_extracted_mvs_total += len(unique_mv_infos)

    if set(pred_mv_norms) == set(gold_mvs):
        s1_exact_sentence_matches += 1

    for norm_mv in pred_mv_norms:
        if norm_mv in gold_lookup:
            s1_correct_mvs_matched += 1

    # --- STAGES 2 & 3 ---
    for mv_info in unique_mv_infos:
        pred_ref = predict_stage2_referent(sentence, mv_info, prompt_word)
        norm_mv = mv_info["normalized_text"]

        gold_match = gold_lookup.get(norm_mv, None)

        if gold_match is not None:
            s2_evaluated_mvs += 1
            if pred_ref == gold_match["referent"]:
                s2_exact_referent_matches += 1

        pred_codes = predict_stage3_codes(sentence, norm_mv, pred_ref)

        if gold_match is not None:
            for col in STAGE3_COLS:
                gold_val = gold_match["codes"][col]
                if gold_val is not None:
                    s3_evaluated_head_counts[col] += 1
                    if pred_codes[col] == gold_val:
                        s3_correct_head_counts[col] += 1

# ============================================================
# PIPELINE FUNNEL & METRICS REPORT
# ============================================================
print("\n" + "=" * 75)
print("                    PIPELINE FUNNEL & ACCURACY METRICS")
print("=" * 75)

print("\n🔻 [FUNNEL TRACKING]")
print(f"  • Total Sentences Input             : {total_val_sentences}")
print(f"  • Sentences with 100% Correct MVs  : {s1_exact_sentence_matches} ({100*s1_exact_sentence_matches/total_val_sentences:.1f}%)")
print(f"  • Total Ground-Truth MVs            : {total_gold_mvs}")
print(f"  • Total MVs Extracted by Stage 1    : {s1_extracted_mvs_total}")
print(f"  • Extracted MVs Matching Ground Truth: {s1_correct_mvs_matched} ({100*s1_correct_mvs_matched/max(s1_extracted_mvs_total,1):.1f}%)")
print(f"  • MVs Advanced to Stage 2 Evaluation: {s2_evaluated_mvs}")

print("\n🎯 [MODEL METRICS]")
print("STAGE 1 (Segmentation):")
print(f"  • Sentence Exact-Set Match Accuracy : {100 * s1_exact_sentence_matches / total_val_sentences:.2f}%")
print(f"  • Individual MV Precision           : {100 * s1_correct_mvs_matched / max(s1_extracted_mvs_total, 1):.2f}%")
print(f"  • Individual MV Recall              : {100 * s1_correct_mvs_matched / total_gold_mvs:.2f}%")

print("\nSTAGE 2 (Referent Extraction):")
print(f"  • Exact Match Referent Accuracy     : {100 * s2_exact_referent_matches / max(s2_evaluated_mvs, 1):.2f}% ({s2_exact_referent_matches}/{s2_evaluated_mvs})")

print("\nSTAGE 3 (Kreitler Psychological Dimensions):")
for col in STAGE3_COLS:
    tot = s3_evaluated_head_counts[col]
    acc = (100 * s3_correct_head_counts[col] / tot) if tot > 0 else 0.0
    print(f"  • Head [{col:8s}] Accuracy          : {acc:.2f}% ({s3_correct_head_counts[col]}/{tot})")
print("=" * 75)

Evaluating Pipeline Funnel: 100%|██████████| 800/800 [01:57<00:00,  6.82it/s]


                    PIPELINE FUNNEL & ACCURACY METRICS

🔻 [FUNNEL TRACKING]
  • Total Sentences Input             : 800
  • Sentences with 100% Correct MVs  : 335 (41.9%)
  • Total Ground-Truth MVs            : 4114
  • Total MVs Extracted by Stage 1    : 3974
  • Extracted MVs Matching Ground Truth: 2856 (71.9%)
  • MVs Advanced to Stage 2 Evaluation: 2856

🎯 [MODEL METRICS]
STAGE 1 (Segmentation):
  • Sentence Exact-Set Match Accuracy : 41.88%
  • Individual MV Precision           : 71.87%
  • Individual MV Recall              : 69.42%

STAGE 2 (Referent Extraction):
  • Exact Match Referent Accuracy     : 73.70% (2105/2856)

STAGE 3 (Kreitler Psychological Dimensions):
  • Head [dim     ] Accuracy          : 77.63% (2214/2852)
  • Head [tr      ] Accuracy          : 86.43% (2465/2852)
  • Head [fr      ] Accuracy          : 96.74% (2759/2852)
  • Head [sr      ] Accuracy          : 88.06% (2515/2856)
  • Head [fe      ] Accuracy          : 99.93% (2851/2853)
  • Head [refLevel] Acc

Another evaluation metrics for comparison

In [ ]:
# ============================================================
# 3-MODEL CASCADE PIPELINE
# ============================================================

import json
import os
import re
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import (
    AutoModel,
    AutoModelForQuestionAnswering,
    AutoModelForTokenClassification,
    AutoTokenizer,
)

# ------------------------------------------------------------
# 1. CONFIGURATION & PATHS
# ------------------------------------------------------------
MODEL_NAME = "dicta-il/dictabert"

STAGE1_PATH = (
    "/content/drive/MyDrive/Workshop/New_Models/stage1_dictabert_fix1/"
)
STAGE2_PATH = "/content/drive/MyDrive/Workshop/New_Models/stage2_dictabert/"
STAGE3_PATH = (
    "/content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty/"
)

VAL_CSV_PATH = "val_set.csv"
TRAIN_CSV_PATH = "train_set.csv"

STAGE3_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("🚀 LOADING 3-MODEL CASCADE EVALUATION PIPELINE")
print(f"Device: {device}")
print("=" * 70)

# ------------------------------------------------------------
# 2. LOAD STAGE 3 LABEL ENCODERS
# ------------------------------------------------------------
s3_encoder_path = os.path.join(STAGE3_PATH, "label_encoders.json")

if os.path.exists(s3_encoder_path):
    print(f"✅ Loading exact label encoders from: {s3_encoder_path}")
    with open(s3_encoder_path, "r", encoding="utf-8") as f:
        encoders = json.load(f)
    decoders = {
        col: {int(v): k for k, v in enc.items()} for col, enc in encoders.items()
    }
    num_classes_dict = {col: len(enc) for col, enc in encoders.items()}
else:
    print(
        "⚠️ label_encoders.json not found in Stage 3 dir. Falling back to sorted CSV keys..."
    )
    ref_df = pd.read_csv(
        TRAIN_CSV_PATH if os.path.exists(TRAIN_CSV_PATH) else VAL_CSV_PATH
    )
    encoders, decoders, num_classes_dict = {}, {}, {}
    for col in STAGE3_COLS:
        values = sorted(ref_df[col].dropna().astype(str).unique().tolist())
        encoders[col] = {value: i for i, value in enumerate(values)}
        decoders[col] = {i: value for value, i in encoders[col].items()}
        num_classes_dict[col] = len(encoders[col])


# ------------------------------------------------------------
# 3. STAGE 3 ARCHITECTURE DEFINITION
# ------------------------------------------------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask


class DictaBertMultiTaskUncertainty(torch.nn.Module):

    def __init__(self, model_name, num_classes_dict):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = torch.nn.Dropout(self.bert.config.hidden_dropout_prob)
        self.heads = torch.nn.ModuleDict({
            col: torch.nn.Linear(hidden_size, num_classes_dict[col])
            for col in STAGE3_COLS
        })
        self.log_vars = torch.nn.Parameter(torch.zeros(len(STAGE3_COLS)))

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = mean_pooling(outputs, attention_mask)
        pooled_output = self.dropout(pooled_output)
        return {col: self.heads[col](pooled_output) for col in STAGE3_COLS}


# ------------------------------------------------------------
# 4. LOAD ALL 3 STAGE MODELS
# ------------------------------------------------------------
# Load Stage 1
print(f"Loading Stage 1 from: {STAGE1_PATH}")
tok_s1 = AutoTokenizer.from_pretrained(STAGE1_PATH)
model_s1 = (
    AutoModelForTokenClassification.from_pretrained(STAGE1_PATH)
    .to(device)
    .eval()
)

# Load Stage 2
print(f"Loading Stage 2 from: {STAGE2_PATH}")
tok_s2 = AutoTokenizer.from_pretrained(STAGE2_PATH)
model_s2 = (
    AutoModelForQuestionAnswering.from_pretrained(STAGE2_PATH)
    .to(device)
    .eval()
)

# Load Stage 3
print(f"Loading Stage 3 from: {STAGE3_PATH}")
model_s3 = DictaBertMultiTaskUncertainty(MODEL_NAME, num_classes_dict)

s3_weights_path = os.path.join(STAGE3_PATH, "pytorch_model.bin")
if not os.path.exists(s3_weights_path):
    s3_weights_path = os.path.join(STAGE3_PATH, "model.safetensors")

state_dict = torch.load(s3_weights_path, map_location=device)
model_s3.load_state_dict(state_dict, strict=False)
model_s3 = model_s3.to(device).eval()
tok_s3 = AutoTokenizer.from_pretrained(STAGE3_PATH)

print("✅ All 3 models loaded successfully.\n")


# ------------------------------------------------------------
# 5. INFERENCE HELPER FUNCTIONS
# ------------------------------------------------------------
def normalize_hebrew(text):
    text = str(text)
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.strip().split())


def predict_stage1_mvs(sentence):
    encoding = tok_s1(
        sentence,
        return_tensors="pt",
        max_length=128,
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
    )
    offset_mapping = encoding.pop("offset_mapping")[0].cpu().tolist()
    inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model_s1(**inputs)

    preds = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    spans, curr_span = [], []

    for idx, (label, (st, end)) in enumerate(zip(preds, offset_mapping)):
        if st == 0 and end == 0:
            continue
        if label == 1:  # B-MU
            if curr_span:
                spans.append(curr_span)
            curr_span = [st, end]
        elif label == 2:  # I-MU
            if curr_span:
                curr_span[1] = end
            else:
                curr_span = [st, end]
        else:
            if curr_span:
                spans.append(curr_span)
                curr_span = []
    if curr_span:
        spans.append(curr_span)

    results = []
    for st, end in spans:
        mv_str = sentence[st:end].strip()
        norm_mv = normalize_hebrew(mv_str)
        if norm_mv:
            results.append({
                "text": mv_str,
                "normalized_text": norm_mv,
                "start_char": st,
                "end_char": end,
            })
    return results


def predict_stage2_referent(sentence, mv_info, prompt_word):
    st, end = mv_info["start_char"], mv_info["end_char"]
    mv_text = mv_info["text"]

    marked_sentence = f"{sentence[:st]}<mv> {mv_text} </mv>{sentence[end:]}"
    context = f"גירוי: {prompt_word} | משפט: {marked_sentence}"
    question = f"מהו הרפרנט של יחידת המשמעות: {mv_text} ?"

    encoding = tok_s2(
        question,
        context,
        max_length=160,
        padding="max_length",
        truncation="only_second",
        return_offsets_mapping=True,
        return_token_type_ids=True,
        return_tensors="pt",
    )
    inputs = {
        k: v.to(device) for k, v in encoding.items() if k != "offset_mapping"
    }

    with torch.no_grad():
        outputs = model_s2(**inputs)

    start_logits = outputs.start_logits[0].cpu().numpy()
    end_logits = outputs.end_logits[0].cpu().numpy()
    token_type_ids = encoding["token_type_ids"][0].cpu().numpy()
    attn = encoding["attention_mask"][0].cpu().numpy()

    # Mask padding and question prompt tokens
    start_logits[(attn == 0) | (token_type_ids == 0)] = -1e9
    end_logits[(attn == 0) | (token_type_ids == 0)] = -1e9

    seq_len = len(start_logits)
    score_matrix = start_logits[:, None] + end_logits[None, :]
    indices = np.arange(seq_len)
    i_idx, j_idx = np.meshgrid(indices, indices, indexing="ij")

    valid_mask = (
        (i_idx <= j_idx)
        & ((j_idx - i_idx + 1) <= 12)
        & (token_type_ids[:, None] == 1)
        & (token_type_ids[None, :] == 1)
    )
    score_matrix[~valid_mask] = -1e9

    best_start, best_end = np.unravel_index(
        np.argmax(score_matrix), score_matrix.shape
    )
    input_ids = encoding["input_ids"][0].cpu().tolist()
    pred_tokens = input_ids[best_start : best_end + 1]
    raw_ref = tok_s2.decode(pred_tokens, skip_special_tokens=True)
    clean_ref = re.sub(r"</?mv>", "", raw_ref)
    return normalize_hebrew(clean_ref)


def predict_stage3_codes(sentence, meaning_val, referent):
    text = f"משפט: {sentence} | יחידה: {meaning_val} | רפרנט: {referent}"
    encoding = tok_s3(
        text,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    inputs = {
        k: v.to(device)
        for k, v in encoding.items()
        if k in ["input_ids", "attention_mask"]
    }

    with torch.no_grad():
        logits = model_s3(**inputs)

    predictions = {}
    for col in STAGE3_COLS:
        pred_idx = torch.argmax(logits[col], dim=-1).item()
        predictions[col] = decoders[col].get(pred_idx, "UNKNOWN")
    return predictions


# ------------------------------------------------------------
# 6. DUAL EVALUATION EXECUTION LOOP
# ------------------------------------------------------------
df_val = pd.read_csv(VAL_CSV_PATH)
grouped = df_val.groupby(["Prompt_Word", "Full_Raw_Sentence"], sort=False)

# Metric Counters
s1_exact_sets, s1_total = 0, 0

s2_oracle_correct, s2_oracle_total = 0, 0
s2_e2e_correct, s2_e2e_total = 0, 0

s3_oracle_correct = {col: 0 for col in STAGE3_COLS}
s3_oracle_total = {col: 0 for col in STAGE3_COLS}

s3_e2e_correct = {col: 0 for col in STAGE3_COLS}
s3_e2e_total = {col: 0 for col in STAGE3_COLS}

for (prompt_word, sentence), sentence_df in tqdm(
    grouped, desc="Processing Sentences"
):
    sentence = str(sentence).strip()
    prompt_word = str(prompt_word).strip() if pd.notna(prompt_word) else ""

    gold_mvs = list(
        dict.fromkeys([
            normalize_hebrew(x)
            for x in sentence_df["MeaningVal"].dropna().astype(str)
        ])
    )

    gold_lookup = {}
    for _, row in sentence_df.iterrows():
        if pd.isna(row["MeaningVal"]):
            continue
        mv_norm = normalize_hebrew(row["MeaningVal"])
        gold_lookup[mv_norm] = {
            "raw_mv": str(row["MeaningVal"]).strip(),
            "referent": normalize_hebrew(row["Referent"])
            if pd.notna(row["Referent"])
            else "",
            "codes": {
                col: None if pd.isna(row[col]) else str(row[col]).strip()
                for col in STAGE3_COLS
            },
        }

    # 1. ORACLE EVALUATION
    for gold_mv_norm, gold_info in gold_lookup.items():
        gold_raw_mv = gold_info["raw_mv"]
        gold_ref = gold_info["referent"]
        gold_pos = sentence.find(gold_raw_mv)

        if gold_pos != -1:
            gold_mv_info = {
                "text": gold_raw_mv,
                "normalized_text": gold_mv_norm,
                "start_char": gold_pos,
                "end_char": gold_pos + len(gold_raw_mv),
            }

            # Stage 2 Oracle
            oracle_ref = predict_stage2_referent(
                sentence, gold_mv_info, prompt_word
            )
            if oracle_ref == gold_ref:
                s2_oracle_correct += 1
            s2_oracle_total += 1

        # Stage 3 Oracle
        oracle_codes = predict_stage3_codes(sentence, gold_mv_norm, gold_ref)
        for col in STAGE3_COLS:
            gold_val = gold_info["codes"][col]
            if gold_val is not None:
                s3_oracle_total[col] += 1
                if oracle_codes[col] == gold_val:
                    s3_oracle_correct[col] += 1

    # 2. END-TO-END CASCADE EVALUATION
    predicted_mv_infos = predict_stage1_mvs(sentence)
    seen = set()
    unique_mv_infos = []

    for item in predicted_mv_infos:
        norm_mv = item["normalized_text"]
        if norm_mv not in seen:
            seen.add(norm_mv)
            unique_mv_infos.append(item)

    pred_mv_texts = [item["normalized_text"] for item in unique_mv_infos]
    if set(pred_mv_texts) == set(gold_mvs):
        s1_exact_sets += 1
    s1_total += 1

    for mv_info in unique_mv_infos:
        pred_ref = predict_stage2_referent(sentence, mv_info, prompt_word)
        pred_mv_norm = mv_info["normalized_text"]
        gold_match = gold_lookup.get(pred_mv_norm)

        s2_e2e_total += 1
        if gold_match is not None:
            if pred_ref == gold_match["referent"]:
                s2_e2e_correct += 1

            # Stage 3 End-to-End
            pred_codes = predict_stage3_codes(
                sentence, pred_mv_norm, pred_ref
            )
            for col in STAGE3_COLS:
                gold_val = gold_match["codes"][col]
                if gold_val is not None:
                    s3_e2e_total[col] += 1
                    if pred_codes[col] == gold_val:
                        s3_e2e_correct[col] += 1

# ------------------------------------------------------------
# 7. FINAL METRIC REPORTING
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("🏆 3-MODEL CASCADE PIPELINE RESULTS")
print("=" * 65)

t1_acc = (s1_exact_sets / max(s1_total, 1)) * 100
print(f"Task 1 — Meaning Value Extraction (Exact Set Match): {t1_acc:.2f}%\n")

print("--- TASK 2: REFERENT EXTRACTION ---")
o_t2 = (s2_oracle_correct / max(s2_oracle_total, 1)) * 100
e_t2 = (s2_e2e_correct / max(s2_e2e_total, 1)) * 100
print(f"  • Oracle Accuracy (Gold MV)         : {o_t2:.2f}%")
print(f"  • End-to-End Cascaded Precision    : {e_t2:.2f}%\n")

print("--- TASK 3: PSYCHOLOGICAL CLASSIFICATION ACCURACY ---")
for col in STAGE3_COLS:
    o_tot = s3_oracle_total[col]
    e_tot = s3_e2e_total[col]
    o_acc = (s3_oracle_correct[col] / o_tot * 100) if o_tot > 0 else 0.0
    e_acc = (s3_e2e_correct[col] / e_tot * 100) if e_tot > 0 else 0.0
    print(
        f"  • {col:<8s} | Oracle: {o_acc:5.2f}% | End-to-End Cascaded:"
        f" {e_acc:5.2f}%"
    )
print("=" * 65)

🚀 LOADING 3-MODEL CASCADE EVALUATION PIPELINE
Device: cuda
✅ Loading exact label encoders from: /content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty/label_encoders.json
Loading Stage 1 from: /content/drive/MyDrive/Workshop/New_Models/stage1_dictabert_fix1/


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Stage 2 from: /content/drive/MyDrive/Workshop/New_Models/stage2_dictabert/


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Stage 3 from: /content/drive/MyDrive/Workshop/New_Models/stage3_dictabert_uncertainty/


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ All 3 models loaded successfully.



Processing Sentences: 100%|██████████| 808/808 [03:46<00:00,  3.57it/s]


🏆 3-MODEL CASCADE PIPELINE RESULTS
Task 1 — Meaning Value Extraction (Exact Set Match): 39.36%

--- TASK 2: REFERENT EXTRACTION ---
  • Oracle Accuracy (Gold MV)         : 71.30%
  • End-to-End Cascaded Precision    : 52.85%

--- TASK 3: PSYCHOLOGICAL CLASSIFICATION ACCURACY ---
  • dim      | Oracle: 78.21% | End-to-End Cascaded: 79.65%
  • tr       | Oracle: 83.86% | End-to-End Cascaded: 85.24%
  • fr       | Oracle: 95.70% | End-to-End Cascaded: 96.14%
  • sr       | Oracle: 92.36% | End-to-End Cascaded: 87.41%
  • fe       | Oracle: 99.93% | End-to-End Cascaded: 99.94%
  • refLevel | Oracle: 74.65% | End-to-End Cascaded: 69.44%
  • ss       | Oracle: 85.19% | End-to-End Cascaded: 87.64%
